## Summarize and check events.

This script summarizes the event structures in the W-H-MEEG dataset
and checks the remapping.  This scripts works on `*_events_temp2.tsv`.


In [4]:
from hed.tools.io_utils import get_file_list, make_file_dict
from hed.tools.data_utils import get_new_dataframe
from hed.tools.hed_logger import HedLogger

bids_root_path = 'G:/WH_working2'
map_path = '../../../data/wakeman_henson_data/wh_map.tsv'
bids_files = get_file_list(bids_root_path, extensions=[".tsv"], name_suffix="_events_temp2")
file_dict = make_file_dict(bids_files, indices=(0, -3))
final_order = ['onset', 'duration', 'sample', 'event_type', 'face_type', 'rep_status',
               'rep_lag', 'trial', 'value', 'stim_file']

status = HedLogger()

# Create teh dictionary
map_dict = {}
map_df = get_new_dataframe(map_path)
for ind, row in map_df.iterrows():
    key = row['value']
    if  key in map_dict:
        status.add(key, f"ERROR {key} duplicated in map dictionary", also_print=True)
    else:
        map_dict[key] = ind

# Check the consistency
for key, file in file_dict.items():
    df = get_new_dataframe(file)
    for ind, row in df.iterrows():
        value =  row['value']
        if value not in map_dict:
            status.add(key, f"ERROR {key} has invalid value {value} in row {ind}", also_print=True)
        elif row['event_type'] != map_df.loc[map_dict[value], 'event_type']:
            status.add(key, f"ERROR {key} has invalid event_type {row['event_type']} in row {ind}", also_print=True)
        elif row['face_type'] != map_df.loc[map_dict[value], 'face_type']:
            status.add(key, f"ERROR {key} has invalid face_type {row['face_type']} in row {ind}", also_print=True)
        elif row['rep_status'] != map_df.loc[map_dict[value], 'rep_status']:
            status.add(key, f"ERROR {key} has invalid frep_status {row['rep_status']} in row {ind}", also_print=True)
        else:
            continue

ERROR sub-002_run-1 has invalid value 4352 in row 155
ERROR sub-004_run-1 has invalid value 4352 in row 72
ERROR sub-004_run-1 has invalid value 4352 in row 583
ERROR sub-004_run-3 has invalid value 4352 in row 508
ERROR sub-005_run-5 has invalid value 4352 in row 373
ERROR sub-007_run-2 has invalid value 4352 in row 52
ERROR sub-009_run-3 has invalid value 4352 in row 528
ERROR sub-014_run-1 has invalid value 4352 in row 163
ERROR sub-014_run-6 has invalid value 4352 in row 275
ERROR sub-014_run-6 has invalid value 4352 in row 441
ERROR sub-016_run-5 has invalid value 4352 in row 498
ERROR sub-016_run-5 has invalid value 4352 in row 502
ERROR sub-017_run-3 has invalid value 4352 in row 115
ERROR sub-019_run-2 has invalid value 4352 in row 118


In [5]:
# print(f"\nBIDS form of the events: {len(file_dict)}")
# status = HedLogger()
# for key, file in file_dict.items():
#     df = get_new_dataframe(file)
#     df.drop(columns=['value', 'trial_type', 'response_time'], inplace=True)
#     status.add(key, f"Dropped the value, trial_type, and response_time columns")
#
#     df.rename(columns={'repetition_type': 'rep_status', 'trigger': 'value'}, inplace=True)
#     status.add(key, f"Renamed repetition_type column as rep_status and trigger column as value")
#
#     show_cross_mask = df['event_type'].map(str) == 'show_cross'
#     df.loc[show_cross_mask, 'value'] = 1
#     status.add(key,f"Set the show_cross value columns to 1")
#
#     df['trial'] = 'n/a'
#     trial = 0
#     for ind, row in df.iterrows():
#         if df.loc[df.index[ind], 'event_type'] == 'show_cross' or \
#             df.loc[df.index[ind], 'event_type'] == 'show_face_initial':
#             trial += 1
#         df.loc[df.index[ind], 'trial'] = trial
#     exclude_mask = (df['event_type'].map(str) == 'setup_left_sym') | \
#                    (df['event_type'].map(str) == 'setup_right_sym')
#     df.loc[exclude_mask, 'trial'] = 'n/a'
#     status.add(key,f"Set the trial numbers")
#
#
#     df['rep_lag'] = 'n/a'
#     immediate_mask = df['rep_status'].map(str) == 'immediate_repeat'
#     df.loc[immediate_mask, 'rep_lag'] = 1
#
#     delayed_mask = df['rep_status'].map(str) == 'delayed_repeat'
#     stim_dict = {}
#     for ind, row in df.iterrows():
#         stim_file = df.loc[ind, 'stim_file']
#         if stim_file == 'n/a':
#             continue
#         elif stim_file not in stim_dict:
#             stim_dict[stim_file] = df.loc[ind, 'trial']
#         elif df.loc[ind, 'rep_status'] == 'delayed_repeat':
#             df.loc[ind, 'rep_lag'] = df.loc[ind, 'trial'] - stim_dict[stim_file]
#     status.add(key,f"Create and fill in the rep_lag column")
#
#     col_list = list(df)
#     if len(col_list) != len(final_order):
#         status.add(key,f"ERROR {key} dataframe has wrong number of columns {len(col_list)}", also_print=True)
#         continue
#     for item in col_list:
#         if item not in final_order:
#            status.add(key,f"ERROR dataframe column {item} should not be there", also_print=True)
#         continue
#     df = df.reindex(columns=final_order)
#     status.add(key,f"Reorder the columns")
#     filename = file[:-10] + "_temp2.tsv"
#     df.to_csv(filename, sep='\t', index=False)
#     status.add(key,f"Save the file as _events_temp.tsv")
#

In [6]:
# status.print_log()
#
# from hed.tools.map_utils import make_combined_dicts
#
# print('\nBIDS events summary:')
# bids_dicts_all, bids_dicts =  make_combined_dicts(bids_file_dict, skip_cols=bids_skip)
# bids_dicts_all.print()